# 01 Recopilación y transformación de datos

Este notebook desarrolla la fase 1 del proyecto: recopilación, extracción, transformación  y carga de los datos climáticos. El objetivo es dejar una base confiable para las fases posteriores de EDA, inteligencia de negocios y modelado predictivo.



## 1. Preparación del entorno

In [20]:
from pathlib import Path
import pandas as pd

# Rutas del proyecto
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_PATH      = ROOT / "data" / "raw"       / "climate_change_dataset.csv"
CLEANED_DATA_PATH  = ROOT / "data" / "cleaned"   / "climate_change_cleaned.csv"
PROCESSED_DATA_PATH= ROOT / "data" / "processed" / "climate_change_model_ready.csv"
REPORTS_PATH       = ROOT / "reports" / "tables"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)


## 2. Fuente de datos

La fuente principal es el archivo `data/raw/climate_change_dataset.csv`, incluido en el repositorio. Contiene indicadores climáticos globales por país y año, con variables como temperatura promedio, emisiones de CO₂, aumento del nivel del mar, precipitación, población, energía renovable, eventos climáticos extremos y área forestal.

In [21]:
# Carga del dataset original
raw_df = pd.read_csv(RAW_DATA_PATH)

## 3. Extracción

Se lee el CSV original sin modificaciones. En esta etapa los datos se presentan tal como están en la fuente.

In [22]:
display(raw_df.head())
print(f"Filas:    {raw_df.shape[0]:,}")
print(f"Columnas: {raw_df.shape[1]}")

,Year,Country,Avg Temperature (°C),CO2 Emissions (Tons/Capita),Sea Level Rise (mm),Rainfall (mm),Population,Renewable Energy (%),Extreme Weather Events,Forest Area (%)
0,2006,UK,8.9,9.3,3.1,1441,530911230,20.4,14,59.8
1,2019,USA,31.0,4.8,4.2,2407,107364344,49.2,8,31.0
2,2014,France,33.9,2.8,2.2,1241,441101758,33.3,9,35.5
3,2010,Argentina,5.9,1.8,3.2,1892,1069669579,23.7,7,17.7
4,2007,Germany,26.9,5.6,2.4,1743,124079175,12.5,4,17.4


Filas:    1,000
Columnas: 10


## 4. Diagnóstico inicial

Revisamos tipos de dato, valores faltantes, duplicados, rango temporal y países incluidos para entender la calidad inicial de la fuente.

In [23]:
diagnostic_summary = pd.DataFrame({
    "tipo_dato"     : raw_df.dtypes.astype(str),
    "missing_values": raw_df.isna().sum(),
    "missing_pct"   : (raw_df.isna().mean() * 100).round(2),
    "unique_values" : raw_df.nunique(dropna=False),
})

display(diagnostic_summary)
print("Duplicados exactos:", int(raw_df.duplicated().sum()))


,tipo_dato,missing_values,missing_pct,unique_values
Year,int64,0,0.0,24
Country,str,0,0.0,15
Avg Temperature (°C),float64,0,0.0,292
CO2 Emissions (Tons/Capita),float64,0,0.0,194
Sea Level Rise (mm),float64,0,0.0,41
Rainfall (mm),int64,0,0.0,799
Population,int64,0,0.0,1000
Renewable Energy (%),float64,0,0.0,407
Extreme Weather Events,int64,0,0.0,15
Forest Area (%),float64,0,0.0,473


Duplicados exactos: 0


In [24]:
raw_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Year,1000.0,NaN,NaN,NaN,2011.432,7.147199,2000.0,2005.0,2012.0,2018.0,2023.0
Country,1000,15,Indonesia,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Avg Temperature (°C),1000.0,NaN,NaN,NaN,19.8831,8.542897,5.0,12.175,20.1,27.225,34.9
CO2 Emissions (Tons/Capita),1000.0,NaN,NaN,NaN,10.4258,5.614665,0.5,5.575,10.7,15.4,20.0
Sea Level Rise (mm),1000.0,NaN,NaN,NaN,3.0096,1.146081,1.0,2.0,3.0,4.0,5.0
Rainfall (mm),1000.0,NaN,NaN,NaN,1738.761,708.976616,501.0,1098.75,1726.0,2362.5,2999.0
Population,1000.0,NaN,NaN,NaN,705383046.613,409390996.884411,3660891.0,343624166.0,713116635.5,1073868037.25,1397016073.0
Renewable Energy (%),1000.0,NaN,NaN,NaN,27.3005,12.970808,5.1,16.1,27.15,38.925,50.0
Extreme Weather Events,1000.0,NaN,NaN,NaN,7.291,4.422655,0.0,3.0,8.0,11.0,14.0
Forest Area (%),1000.0,NaN,NaN,NaN,40.572,17.398998,10.1,25.6,41.15,55.8,70.0


## 5. Transformación

La transformación realiza los siguientes pasos:
- Renombra las columnas al español con nombres estandarizados
- Limpia espacios en variables de texto
- Asegura los tipos de dato correctos
- Ordena las filas por país y año
- Elimina duplicados exactos si los hubiera

No se imputan datos porque la fuente no presenta valores faltantes.

In [25]:
# Paso 1: renombrar columnas
COLUMN_MAP = {
    "Year"                      : "Anio",
    "Country"                   : "Pais",
    "Avg Temperature (°C)"      : "Promedio_temperatura_c",
    "CO2 Emissions (Tons/Capita)": "Emisiones_CO2_TonXHab",
    "Sea Level Rise (mm)"        : "Aumento_Nivel_Mar_mm",
    "Rainfall (mm)"              : "Precipitaciones_mm",
    "Population"                 : "Poblacion",
    "Renewable Energy (%)"       : "Energia_renovable_Pct",
    "Extreme Weather Events"     : "Eventos_Extremos",
    "Forest Area (%)"            : "Area_Forestal_Pct",
}

cleaned_df = raw_df.rename(columns=COLUMN_MAP).copy()

# Paso 2: limpiar espacios en texto
cleaned_df["Pais"] = cleaned_df["Pais"].str.strip()

# Paso 3: asegurar tipos de dato
cleaned_df["Anio"]                   = cleaned_df["Anio"].astype(int)
cleaned_df["Poblacion"]              = cleaned_df["Poblacion"].astype(int)
cleaned_df["Eventos_Extremos"]       = cleaned_df["Eventos_Extremos"].astype(int)
cleaned_df["Promedio_temperatura_c"] = cleaned_df["Promedio_temperatura_c"].astype(float)
cleaned_df["Emisiones_CO2_TonXHab"]  = cleaned_df["Emisiones_CO2_TonXHab"].astype(float)
cleaned_df["Aumento_Nivel_Mar_mm"]   = cleaned_df["Aumento_Nivel_Mar_mm"].astype(float)
cleaned_df["Precipitaciones_mm"]     = cleaned_df["Precipitaciones_mm"].astype(float)
cleaned_df["Energia_renovable_Pct"]  = cleaned_df["Energia_renovable_Pct"].astype(float)
cleaned_df["Area_Forestal_Pct"]      = cleaned_df["Area_Forestal_Pct"].astype(float)

# Paso 4: orden de columnas
FINAL_COLUMNS = [
    "Anio", "Pais",
    "Promedio_temperatura_c", "Emisiones_CO2_TonXHab",
    "Energia_renovable_Pct", "Eventos_Extremos",
    "Precipitaciones_mm", "Area_Forestal_Pct",
    "Aumento_Nivel_Mar_mm", "Poblacion",
]
cleaned_df = cleaned_df[FINAL_COLUMNS]

# Paso 5: ordenar filas y eliminar duplicados
cleaned_df = (
    cleaned_df
    .drop_duplicates()
    .sort_values(["Pais", "Anio"])
    .reset_index(drop=True)
)

display(cleaned_df.head(10))
print("\nTipos de dato finales:")
display(cleaned_df.dtypes.to_frame("tipo_dato"))


,Anio,Pais,Promedio_temperatura_c,Emisiones_CO2_TonXHab,Energia_renovable_Pct,Eventos_Extremos,Precipitaciones_mm,Area_Forestal_Pct,Aumento_Nivel_Mar_mm,Poblacion
0,2000,Argentina,16.9,3.9,15.5,11,2047.0,18.4,4.0,564877556
1,2001,Argentina,33.2,18.2,15.7,1,2372.0,39.7,2.9,1078565697
2,2001,Argentina,24.8,14.3,33.8,12,2813.0,15.6,2.8,332640493
3,2001,Argentina,21.5,1.1,31.6,2,2005.0,17.7,4.7,597841637
4,2001,Argentina,8.4,16.8,12.7,3,2035.0,19.8,2.3,1334032802
5,2002,Argentina,14.3,16.6,26.7,7,1325.0,58.2,2.6,845772793
6,2002,Argentina,27.6,13.6,11.1,10,1056.0,23.8,1.6,711474124
7,2003,Argentina,10.0,16.5,21.9,13,626.0,17.2,1.3,1211429158
8,2004,Argentina,9.8,2.1,15.6,1,1290.0,40.1,1.5,742981303
9,2004,Argentina,15.4,8.5,31.8,0,1416.0,46.9,2.0,1114422920



Tipos de dato finales:


,tipo_dato
Anio,int64
Pais,str
Promedio_temperatura_c,float64
Emisiones_CO2_TonXHab,float64
Energia_renovable_Pct,float64
Eventos_Extremos,int64
Precipitaciones_mm,float64
Area_Forestal_Pct,float64
Aumento_Nivel_Mar_mm,float64
Poblacion,int64


In [26]:
# Reporte de calidad general
quality_report = pd.DataFrame({
    "tipo_dato"     : cleaned_df.dtypes.astype(str),
    "missing_values": cleaned_df.isna().sum(),
    "missing_pct"   : (cleaned_df.isna().mean() * 100).round(2),
    "unique_values" : cleaned_df.nunique(),
    "min"           : cleaned_df.min(numeric_only=True),
    "max"           : cleaned_df.max(numeric_only=True),
})
display(quality_report)


,tipo_dato,missing_values,missing_pct,unique_values,min,max
Anio,int64,0,0.0,24,2000.0,2.023000e+03
Area_Forestal_Pct,float64,0,0.0,473,10.1,7.000000e+01
Aumento_Nivel_Mar_mm,float64,0,0.0,41,1.0,5.000000e+00
Emisiones_CO2_TonXHab,float64,0,0.0,194,0.5,2.000000e+01
Energia_renovable_Pct,float64,0,0.0,407,5.1,5.000000e+01
Eventos_Extremos,int64,0,0.0,15,0.0,1.400000e+01
Pais,str,0,0.0,15,NaN,NaN
Poblacion,int64,0,0.0,1000,3660891.0,1.397016e+09
Precipitaciones_mm,float64,0,0.0,799,501.0,2.999000e+03
Promedio_temperatura_c,float64,0,0.0,292,5.0,3.490000e+01


## 7. Dataset procesado para fases siguientes

Además del dataset limpio, se genera una versión procesada con variables derivadas para BI y análisis posterior:
- Continente: continente de cada país
- Decada: década del registro (ej. 2000s, 2010s, 2020s)
- Nivel_energia_renovable: categoría Bajo / Medio / Alto
- Categoria_temperatura: categoría Fría / Templada / Cálida / Muy cálida


In [27]:
processed_df = cleaned_df.copy()

# Variable derivada 1: Continente
CONTINENTE_MAP = {
    "Argentina"   : "América del Sur",
    "Australia"   : "Oceanía",
    "Brazil"      : "América del Sur",
    "Canada"      : "América del Norte",
    "China"       : "Asia",
    "France"      : "Europa",
    "Germany"     : "Europa",
    "India"       : "Asia",
    "Indonesia"   : "Asia",
    "Japan"       : "Asia",
    "Mexico"      : "América del Norte",
    "Russia"      : "Europa",
    "South Africa": "África",
    "UK"          : "Europa",
    "USA"         : "América del Norte",
}
processed_df["Continente"] = processed_df["Pais"].map(CONTINENTE_MAP)

# Variable derivada 2: Década
processed_df["Decada"] = (processed_df["Anio"] // 10 * 10).astype(str) + "s"

# Variable derivada 3: Nivel de energía renovable
def clasificar_energia(pct):
    if pct < 20:
        return "Bajo"
    elif pct < 40:
        return "Medio"
    else:
        return "Alto"

processed_df["Nivel_energia_renovable"] = processed_df["Energia_renovable_Pct"].apply(clasificar_energia)

# Variable derivada 4: Categoría de temperatura
def clasificar_temperatura(temp):
    if temp < 10:
        return "Fría"
    elif temp < 20:
        return "Templada"
    elif temp < 30:
        return "Cálida"
    else:
        return "Muy cálida"

processed_df["Categoria_temperatura"] = processed_df["Promedio_temperatura_c"].apply(clasificar_temperatura)

# Reordenar columnas
processed_df = processed_df[[
    "Anio", "Decada", "Pais", "Continente",
    "Promedio_temperatura_c", "Categoria_temperatura",
    "Emisiones_CO2_TonXHab", "Energia_renovable_Pct", "Nivel_energia_renovable",
    "Eventos_Extremos", "Precipitaciones_mm",
    "Area_Forestal_Pct", "Aumento_Nivel_Mar_mm", "Poblacion",
]]

display(processed_df.head(10))


,Anio,Decada,Pais,Continente,Promedio_temperatura_c,Categoria_temperatura,Emisiones_CO2_TonXHab,Energia_renovable_Pct,Nivel_energia_renovable,Eventos_Extremos,Precipitaciones_mm,Area_Forestal_Pct,Aumento_Nivel_Mar_mm,Poblacion
0,2000,2000s,Argentina,América del Sur,16.9,Templada,3.9,15.5,Bajo,11,2047.0,18.4,4.0,564877556
1,2001,2000s,Argentina,América del Sur,33.2,Muy cálida,18.2,15.7,Bajo,1,2372.0,39.7,2.9,1078565697
2,2001,2000s,Argentina,América del Sur,24.8,Cálida,14.3,33.8,Medio,12,2813.0,15.6,2.8,332640493
3,2001,2000s,Argentina,América del Sur,21.5,Cálida,1.1,31.6,Medio,2,2005.0,17.7,4.7,597841637
4,2001,2000s,Argentina,América del Sur,8.4,Fría,16.8,12.7,Bajo,3,2035.0,19.8,2.3,1334032802
5,2002,2000s,Argentina,América del Sur,14.3,Templada,16.6,26.7,Medio,7,1325.0,58.2,2.6,845772793
6,2002,2000s,Argentina,América del Sur,27.6,Cálida,13.6,11.1,Bajo,10,1056.0,23.8,1.6,711474124
7,2003,2000s,Argentina,América del Sur,10.0,Templada,16.5,21.9,Medio,13,626.0,17.2,1.3,1211429158
8,2004,2000s,Argentina,América del Sur,9.8,Fría,2.1,15.6,Bajo,1,1290.0,40.1,1.5,742981303
9,2004,2000s,Argentina,América del Sur,15.4,Templada,8.5,31.8,Medio,0,1416.0,46.9,2.0,1114422920


## 8. Carga de resultados

Se guardan los dos datasets (limpio y procesado)

In [28]:

# Guardar archivos
cleaned_df.to_csv(CLEANED_DATA_PATH, index=False)
processed_df.to_csv(PROCESSED_DATA_PATH, index=False)


## 10. Resultados de la fase 1
- Diagnóstico general de los datos.
- Transformación de los datos.
- Agregamos variables como continente, Década, Nivel_energia_renovable y Categoria_temperatura
para modelado posterior.
- Finalmente, guardamos los archivos con sus respectivos cambios.